# 4주차 실습: 한글 형태소 분석과 토큰화

**주제:** Kiwi·KoNLPy(Okt)를 이용한 형태소 분석, 품사 태깅, 사용자 사전

## 학습 목표
1. 어절·형태소·토큰의 차이를 설명한다.
2. 공백 기반 토큰화와 형태소 분석 기반 토큰화를 비교한다.
3. Kiwi와 Okt로 품사 태깅을 수행하고 과업에 맞는 분석기를 판단한다.
4. 사용자 사전이 필요한 사례를 만들고 적용한다.

> **제출:** 모든 TODO를 작성·실행하고, 체크포인트 답변을 Markdown 셀에 남기세요. 외부 AI·자료를 사용했다면 사용 범위와 검증 과정을 기록하세요.

## 0. 준비
교육용 예시 데이터입니다. 실제 데이터는 출처·라이선스·개인정보 포함 여부를 확인하세요.

In [ ]:
# 실행 오류가 날 때만 아래 주석을 해제하세요.
!pip -q install -U kiwipiepy konlpy

import re
import pandas as pd # 표 형태 출력 위함
from kiwipiepy import Kiwi
from konlpy.tag import Okt

kiwi = Kiwi()
okt = Okt()

texts = [
    '충북대학교 자연어처리 수업이 정말 재미있어요!',
    'AI 챗봇의 답변 품질을 개선하고 싶어요.',
    '배터리가 오래가고 화면도 선명해서 만족합니다.',
    '결말은 아쉬웠지만 배우 연기는 최고였어요.',
    '새로 나온 갓생 챌린지 앱을 사용해 봤어요.'
]
df = pd.DataFrame({'text_id': range(1, len(texts)+1), 'text': texts})
display(df)

## 1. 어절·형태소·토큰
- **어절:** 띄어쓰기로 구분되는 단위
- **형태소:** 뜻 또는 문법 기능을 지닌 최소 단위
- **토큰:** 분석 목적에 맞게 모델 입력으로 정한 단위

예: `학생들이 도서관에서 공부한다.` → `학생/들/이`, `도서관/에서`, `공부/하/ㄴ다`처럼 분해할 수 있습니다.

In [ ]:
sample = '학생들이 도서관에서 공부한다.'
print('원문:', sample)

# sol1. 공백 기준 토큰화
print('공백 토큰:', sample.split())

# sol2. 정규표현식 기준 토큰화
print('정규표현식 토큰:', re.findall(r'[가-힣A-Za-z0-9]+', sample))

# sol3. 형태소 기준 토큰화(한국어는 접사가 붙어 기능인 품사가 바뀌는 교착어의 특성상 형태소 분석이 필요)

# TODO 1: Kiwi 형태소 분석 결과를 [(형태, 품사), ...] 형태로 출력하세요.
kiwi_result = [(token.form, token.tag) for token in kiwi.tokenize(sample)]
print(kiwi_result)

# Kiwi는 세종 태그셋을 기반으로 하며, 용언의 규칙성 표기, 인터넷 어휘 처리를 위한 확장 태그 체계가 추가됨
# 학생(NNG): 일반 명사
# 들(XSN): 명사 파생 접미사(여기선 복수형이라는 의미 가짐)
# 이(JKS): 주격조사(주어를 수식)
# 도서관(NNG): 일반 명사
# 에서(JKB): 부사격조사(장소)
# 공부(NNG): 일반 명사
# 하(XSV): 동사 파생 접미사(명사를 동사로 바꿔주는 역할)
# ㄴ다(EF): 현재 시제 평서형 종결어미(Ending Final)
# .(SF): 종결어미 뒤에 붙이는 문장 종결 부호(Sentence-Final)

### 체크포인트 1
공백 토큰화로는 보이지 않는 문법·의미 정보 한 가지를 위 문장에서 찾아 1~2문장으로 설명하세요.

- 현재 시제 평서형 종결어미 'ㄴ다'를 통해 현재 상황임을 파악할 수 있습니다. __*(AI)*__
- '학생들이'라는 토큰이 아닌, '학생'이라는 의미적 본질을 추출할 수 있습니다. __*(AI)*__

## 2. Kiwi로 형태소 분석과 품사 태깅
Kiwi는 현대 한국어 분석에 활용하기 좋은 형태소 분석기입니다. `token.form`은 형태, `token.tag`는 품사 태그입니다.

In [ ]:
def kiwi_pos(text):
    return [(token.form, token.tag) for token in kiwi.tokenize(text)]

for text in df['text']:
    print(text)
    print(kiwi_pos(text))
    print()

# TODO 2: 아래 문장의 명사(태그가 N으로 시작)만 추출하는 함수를 작성하세요.
def kiwi_nouns(text):
    return [
        token.form
        for token in kiwi.tokenize(text) # 토큰화한 뒤
          if token.tag.startswith('N') # 태그가 N으로 시작하는, 즉 명사만 추출
    ]

print(kiwi_nouns(df.loc[0, 'text'])) # 첫번째 행의 text에 대해 적용

## 3. KoNLPy의 Okt로 형태소 분석
Okt는 소셜 텍스트·이모티콘·비표준 표현을 살펴볼 때 자주 사용합니다. 분석기마다 분리 방식과 품사 체계가 다를 수 있습니다.

In [ ]:
print(okt.tagset) # 품사 태그와 한글 설명 딕셔너리 설명 RECEIVED HELP FROM AI

# 'Adjective': '형용사'           'Adverb': '부사'
# 'Alpha': '알파벳'               'Conjunction': '접속사'
# 'Determiner': '관형사'          'Eomi': '어미'
# 'Exclamation': '감탄사'         'Foreign': '외국어, 한자 및 기타기호'
# 'Hashtag': '트위터 해쉬태그'    'Josa': '조사'
# 'KoreanParticle': '(ex: ㅋㅋ)'  'Noun': '명사'
# 'Number': '숫자'                'PreEomi': '선어말어미'
# 'Punctuation': '구두점'         'ScreenName': '트위터 아이디'
# 'Suffix': '접미사'              'Unknown': '미등록어'
# 'Verb': '동사'

compare_text = '오늘 수업 진짜 재밌었는데 과제는 조금 어렵네요 ㅠㅠ'
print('원문:', compare_text)
print('Kiwi:', kiwi_pos(compare_text))
print('Okt :', okt.pos(compare_text))

# TODO 3: 아래 문장을 Kiwi와 Okt로 각각 분석하고 결과 차이를 관찰하세요.
my_sentence = '신조어와 줄임말도 분석할 수 있을까?'
print('Kiwi:', kiwi_pos(my_sentence))
print('Okt :', okt.pos(my_sentence))

print('Kiwi:', kiwi_pos(df.loc[4, 'text']))
print('Okt :', okt.pos(df.loc[4, 'text']))

# Kiwi vs Okt
# NNG vs Noun
# MAG vs Noun
# VA + EP + EC vs Adjective
# JX vs Josa
# SW vs KoreanParticle
# NNG + NNG + JX vs Verb
# XSV + ETM vs Verb

# 세종 태그를 이용하면 NNG식으로 의미를 부여하게 됨
# Okt는 의미가 명확한 태그가 아니라, 다르게 달린 게 있음

# 복사해서 ai에 넣고 태그 분석해달라고 하면 설명해줌 ex. MAG(진짜) -> 부사, JKO(을) -> 목적격 조사, VA-I(아쉽) -> 불규칙 형용사(붙는 어미에 따라 변함. Irregular), E~ -> 어미, XPN(생) -> 체언 접두사(ex. 생과일)(갓생을 잘못 분류한듯..)


### 체크포인트 2
Kiwi와 Okt 결과에서 토큰 분리 또는 품사 태깅이 다른 사례를 하나 기록하세요. 어느 분석기가 “더 좋다”가 아니라, 어떤 **데이터/과업**에서 더 편리할지 설명하세요.

- '진짜'를 Kiwi는 Mag(부사)로, Okt는 Noun(명사)로 태깅합니다.
- '갓생'의 '생'을 Kiwi는 XPN(체언 접두사)로, Okt는 Noun(명사)로 태깅합니다.
---
- Kiwi는 빠른 처리 속도를 자랑하며, 웹이나 문어체 등을 처리하는 데 적합합니다. 따라서 대형 언어 모델과 연동하기 위해 긴 문서를 문장 단위로 정확하게 쪼개거나 텍스트를 파싱할 때 편리합니다.__*(AI)*__
- KoNLPy의 Okt는 신조어, 줄임말, 이모티콘 등이 포함된 비정형 텍스트 분석에 강점이 있습니다. 따라서 긍정/부정을 판별하는 감성 분석에 편리합니다.__*(AI)*__

## 4. 과업에 따른 품사 선택
키워드 추출은 명사 중심 처리가 유용할 수 있습니다. 반면 감성 분석에서는 형용사·동사·부정 표현·강조 부사도 중요한 신호가 됩니다.

In [ ]:
review = '배송은 조금 느렸지만 제품 디자인이 매우 예쁘고 만족스러워요.'
print('전체 분석:', kiwi_pos(review))

# TODO 4: 명사(N*)와 용언(V*)을 각각 추출하세요.
def select_pos(text):
    nouns = []
    predicates = []
    for token in kiwi.tokenize(text):
        if token.tag.startswith('N'):
          nouns.append(token.form)
        elif token.tag.startswith('V'):
          predicates.append(token.form)
    return nouns, predicates

nouns, predicates = select_pos(review)
print('명사:', nouns)
print('용언:', predicates)

### 체크포인트 3
`좋지 않아요`, `매우 예쁘고` 같은 표현을 생각해 보세요. 명사만 남기면 감성 분석에서 어떤 정보가 사라질까요?

- 감정을 직접적으로 표현하는 형용사, 동사, 부사 및 일부 감탄사가 빠져 감정이 빠집니다.

## 5. 사용자 사전: 왜 필요한가
새 서비스명, 과목명, 브랜드명, 전문 용어는 일반 사전에 없거나 원하지 않는 방식으로 분리될 수 있습니다. 사용자 사전은 **도메인 용어를 원하는 품사와 단위로 분석**하도록 돕습니다.

아래 코드는 Kiwi 버전에 따라 사용자 단어 추가 방식이 다를 수 있으므로, 실행 환경의 오류 메시지와 라이브러리 문서를 함께 확인하세요.

In [ ]:
domain_text = '충북대학교에서 자연어처리, 머신 러닝, 딥러닝, 갓생 살기, 데이터마이닝 프로젝트를 진행한다.'
print('추가 전:', kiwi_pos(domain_text))

# TODO 5: 사용자 단어를 추가한 뒤 다시 분석하세요.
kiwi.add_user_word('자연어처리', 'NNP')
kiwi.add_user_word('충북대학교', 'NNP')
kiwi.add_user_word('머신 러닝', 'NNP')
kiwi.add_user_word('딥러닝', 'NNP')
kiwi.add_user_word('갓생', 'NNP')
kiwi.add_user_word('데이터마이닝', 'NNP')
print('추가 후:', kiwi_pos(domain_text))

# 효과
# 1. 개체명 인식(NER) 성능 향상: 핵심 개체 정보의 손실 방지
# 2. 모델 학습 효율 증대: 불필요하게 쪼개지는 토큰 개수를 줄여 전체 시퀀스 길이를 줄이고 모델의 어휘 사전 관리 부담을 완화
# 3. 도메인 지식 주입: 도메인 특화 단어, 신조어, 전문 용어를 직접 반영하여 다운스트림(사용자 측) NLP 작업의 분석 정밀도를 높임

# TODO 6: 본인의 전공/관심 분야 고유명사 2개를 추가하고, 전후 결과를 비교하세요.

## 6. 미니 과제: 토큰화 정책 설계
다음 목표 중 하나를 선택해 토큰화 정책을 설계하세요.
1. 강의평 키워드 추출
2. 영화 리뷰 감성 분석
3. 학교 민원 자동 분류

아래 함수에 분석기, 보존할 품사, 제거할 토큰의 기준을 작성하세요.

In [ ]:
# 데이터: 일기 데이터
text_list = [ # 데이터를 찾아왔다고 가정
    '주말 내내 밀린 잠을 청했는데도 월요일 아침의 피로는 어쩔 수 없나 보다.',
    '매콤한 크림 파스타가 기대 이상으로 맛있어서 오전의 스트레스가 조금은 풀리는 기분이었다.',
    '오후에는 밀린 자연어 처리 과제를 하느라 정신없이 시간이 흘러갔다.'
]

df1 = pd.DataFrame({'text_id': range(1, len(text_list)+1), 'text': text_list})
display(df1)

kiwi.add_user_word('크림 파스타', 'NNP', score = 20) # 단어가 통째로 인식되게 하기 위해 score값을 높여서 등록. RECEIVED HELP FROM AI

def tokenize_for_task_kiwi(text):
    """TODO 7: 선택한 과업에 맞는 토큰화 규칙을 구현하세요.
    - 어떤 품사를 남길지
    - 조사/기호/URL 등을 어떻게 다룰지
    - 부정 표현·이모지·반복 문자를 보존할지
    """
    nouns = []
    predicates = []
    for token in kiwi.tokenize(text): # kiwi는 임시. okt로 변경 예정(일기 데이터셋이므로, 정확한 텍스트보다는 신조어, 구어체 등이 포함될 가능성이 높기 때문)
      if token.tag.startswith('N'): # 키워드 분석을 위해 명사를 남김
        nouns.append(token.form)
      elif token.tag.startswith('V'): # 감성 분석을 위해 용언을 남김
        predicates.append(token.form)

    return nouns, predicates

def tokenize_for_task_okt(text):
  nouns = []
  predicates = []

  pos_list = okt.pos(text)
  for pos in pos_list:
    tag = pos[1]
    if tag == 'Noun':
      nouns.append(pos[0])
    elif tag == 'Adjective' or tag == 'Verb':
      predicates.append(pos[0])

  return nouns, predicates

print('[Kiwi]')
for text in df1['text']:
    nouns, predicates = tokenize_for_task_kiwi(text)
    print(text, '->', nouns, ' & ', predicates)

print('\n[Okt]')
for text in df1['text']:
  nouns, predicates = tokenize_for_task_okt(text)
  print(text, '->', nouns, ' & ', predicates)

### 체크포인트 4
선택한 과업, 분석기 선택 이유, 보존/제거 규칙, 예상되는 오류 사례를 각각 1문장 이상 작성하세요.

- 선택한 과업은 일기 데이터를 분석하여 그날 있었던 일들을 정리하고, 감정을 정리하여 추후 볼 수 있도록 요약하는 것이다. 예를 들어, 크림 파스타를 먹었던 날이라든가, 우울했던 주라든가를 정리하는 식이다.
- 일기 데이터셋이므로, 정확한 텍스트보다는 신조어, 구어체 등이 포함될 가능성이 높기에 okt 분석기를 사용할 예정이다.
- 감성 분석도 진행하기에, 조사와 특수기호 등은 생략하지 않는다.
- 개인 일기이기에, 신조어나 줄임말, 본인만의 표현이 등장하여 형태소 분석에 어려움이 있을 것으로 보인다. 사용자 사전을 체계적으로 관리할 필요가 있어보인다.

**형성평가(Quick Quiz)**

#### Q1. 한국어처럼 실질 형태소에 형식 형태소가 결합하여 문법적 기능을 나타내는 언어 체계를 (**교착**)어라고 합니다.

#### Q2. '아이브(IVE)'와 같은 신조어 오분석을 막기 위해 형태소 분석기에 추가하는 기능은 **사용자 정의 사전**입니다.


## 제출 체크리스트
- [x] TODO 1~7을 작성하고 실행했다.
- [x] 공백·정규표현식·형태소 기반 토큰화를 비교했다.
- [x] Kiwi와 Okt의 결과 차이 및 과업 적합성을 설명했다.
- [x] 명사 중심 처리의 장점과 한계를 설명했다.
- [x] 사용자 사전에 고유명사 2개 이상을 추가하고 전후 결과를 비교했다.
- [x] 선택 과업에 대한 토큰화 정책과 오류 사례를 기록했다.
- [x] 외부 AI/자료 사용 시 범위와 검증 과정을 기록했다.

## 다음 주차 예고
전처리된 토큰을 Bag-of-Words, TF-IDF, n-gram으로 수치화하여 문서 비교와 분류 모델의 입력 특징을 구성합니다.